1. Instalar dependencias

In [1]:
!pip install -q sentence-transformers pandas pyarrow

2. Imports y rutas de los archivos

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import torch

RUTA_BASE      = "/content/drive/MyDrive/Semestre IX/TABD/Articulo TABD"
PARQUET_EMBED  = f"{RUTA_BASE}/Procesado-DataSet-Obras-Publicas 04-06-2026.parquet"
PARQUET_OUTPUT = f"{RUTA_BASE}/embeddings_output.parquet"

print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"Dispositivo    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Mounted at /content/drive
GPU disponible : True
Dispositivo    : Tesla T4


3. Cargar datos

In [3]:
import os
if not os.path.exists(PARQUET_EMBED):
    raise FileNotFoundError(
        f"No se encuentra el parquet en:\n{PARQUET_EMBED}\n"
        "Verifica que la carpeta compartida esté en 'Mi unidad' (no solo compartida)."
    )
df = pd.read_parquet(PARQUET_EMBED, columns=["Código INFOBRAS", "texto_embedding"])
print(f"Registros cargados: {len(df):,}")
print(df["texto_embedding"].iloc[0][:300])

Registros cargados: 191,180
LA OBRA "IMPLEMENTACION DE LA CAPACIDAD RESOLUTIVA DE LA UNIDAD DE GINECO OBSTETRICIA 2DO NIVEL HOSPITAL DE CHANCAY Y SERVICIOS BASICOS DE SALUD", DE NATURALEZA MEJORAMIENTO, CORRESPONDE A SALUD > ATENCION MEDICA > OTRA INFRAESTRUCTURA. EJECUTADA BAJO MODALIDAD DE ADMINISTRACION DIRECTA POR HOSPITAL


4. Generar embeddings

In [4]:
MODELO_ID  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
BATCH_SIZE = 256   # T4 16GB aguanta 256 sin problema; bajar a 128 si da OOM
DIMENSIONES = 384  # fijas del modelo
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = SentenceTransformer(MODELO_ID, device=device)
textos = df["texto_embedding"].tolist()
print(f"Generando {len(textos):,} embeddings en {device.upper()} ...")
vectores = model.encode(
    textos,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,   # cosine
    convert_to_numpy=True,
)
print(f"Shape resultante: {vectores.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando 191,180 embeddings en CUDA ...


Batches:   0%|          | 0/747 [00:00<?, ?it/s]

Shape resultante: (191180, 384)


5. Exportar

In [5]:
cols_vec = [f"v{i}" for i in range(DIMENSIONES)]
df_vec   = pd.DataFrame(vectores.astype("float32"), columns=cols_vec)

df_output = pd.concat([
    df["Código INFOBRAS"].reset_index(drop=True),
    df_vec
], axis=1)

df_output.to_parquet(PARQUET_OUTPUT, index=False, compression="snappy")

# Reporte de tamaño
import os
mb = os.path.getsize(PARQUET_OUTPUT) / 1024 / 1024
print(f"Archivo guardado : {PARQUET_OUTPUT}")
print(f"Tamaño en disco  : {mb:.1f} MB")
print(f"Shape final      : {df_output.shape}")

Archivo guardado : /content/drive/MyDrive/Semestre IX/TABD/Articulo TABD/embeddings_output.parquet
Tamaño en disco  : 435.7 MB
Shape final      : (191180, 385)
